This variation of BC+PPO is strictly based on the paper and implementation of "Bootstrapping Reinforcement Learning with Imitation for Vision-Based Agile Flight" by UZH.
Kindly refer to the paper at: https://arxiv.org/pdf/2403.12203

In [11]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from stable_baselines3.common.buffers import ReplayBuffer
from stable_baselines3 import PPO
import torch.nn as nn
import torch.optim as optim
from torch.distributions.categorical import Categorical


In [12]:
from pandas.core.arrays import categorical
class ActrorCritic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super.__init__()
        self.actor = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, action_dim)
        )
        self.critic= nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
        )

        def get_value(self, x):
            return self.critic(x)
        
        def get_action_value(self, x, action = None):
            logits = self.actor(x)
            probs = Categorical(logits = logits)
            if action is None:
                action = probs.sample()
            return action, probs.log_prob(action), probs.entropy(), self.critic(x)

In [13]:
def Expert(state):
    a = state[2]
    b = state[3]
    return 1 if (a + 0.25 * b) > 0 else 0

In [14]:
from numpy import dtype
def get_expert_data(env, replay_buffer, num_tuples = 10_000):
    state, _ = env.reset()
    for _ in range(num_tuples):
        action = Expert(state)
        next_state, reward, terminated, truncated, info = env.step(action)
        action_array = np.array([action])
        reward_array = np.array([reward])
        done_array = np.array([terminated or truncated], dtype = np.float32)
        replay_buffer.add(
            obs = state,
            next_obs = next_state,
            action = action_array,
            reward = reward_array,
            done = done_array,
            infos =[info]
        )
        if terminated or truncated:
            state, _ = env.reset()
        else:
            state = next_state

In [15]:
def warm_start(agent, replay_buffer, lr = 1e-4, epochs = 20, batch_size = 64, num_samples = 2000):
    print(f"Starting the training now (for {epochs} epochs)")
    optimizer = optim.Adam(agent.actor.parameters(), lr = lr)
    loss_fn = nn.CrossEntropyLoss()
    iterations_per_epoch = num_samples // batch_size
    total_steps = epochs*iterations_per_epoch

    for step in range(total_steps):
        samples = replay_buffer.sample(batch_size = batch_size)
        batch_states = samples.observations
        batch_actions = samples.action.flatten().long()
        logits = agent.actor(batch_states)
        loss = loss_fn(logits, batch_states)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step%iterations_per_epoch == 0:
            current_epoch = step // iterations_per_epoch+1
            print(f"Current Warm-Start epoch {current_epoch} out of {epochs} epochs")
    print('Warm up done')

In [ ]:
def train_PPO(env, agent, total_timesteps = 1_000_000, num_steps = 2048, gae_lambda = 0.95, gamma = 0.99, ppo_epochs = 25, mini_batch_size = 64, clip_coeff = 0.2, ent_coeff = 0.001, vf_coeff = 0.5, lr = 1e-4):

    optimizer = optim.Adam(agent.parameters(), lr = lr)
    states = torch.zeros((num_steps, env.observation_space[0]))
    actions = torch.zeros(num_steps)
    logprobs = torch.zeros(num_steps)
    rewards = torch.zeros(num_steps)
    dones = torch.zeros